# Week 6, Lab 5 — Capstone

1. Use both MCP servers.
2. Answer math + facts + save a study note.
3. Guardrail: refuse `password` / `api key`.
4. Write a short reflection on frameworks vs MCP.


In [1]:
WEEK = 'Week 6'
LAB = 'Lab 5 — capstone'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


Week 6 / Lab 5 — capstone
Backend: ollama
Need Ollama running: `ollama serve` and `ollama pull llama3.2:1b`
If import failed, unzip/clone the WHOLE course folder (not a single notebook).


In [3]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

TOOLS = StdioServerParameters(command="python", args=[str(ROOT / "6_mcp" / "servers" / "local_tools_server.py")])
NOTES = StdioServerParameters(command="python", args=[str(ROOT / "6_mcp" / "servers" / "notes_server.py")])
ROUTING = {
    "calculator": TOOLS,
    "lookup_fact": TOOLS,
    "add_note": NOTES,
    "list_notes": NOTES,
}

def blocked(text: str) -> bool:
    t = text.lower()
    return "password" in t or "api key" in t or "api_key" in t

async def call_tool(name: str, args: dict):
    params = ROUTING[name]
    async with stdio_client(params) as (r, w):
        async with ClientSession(r, w) as s:
            await s.initialize()
            return await s.call_tool(name, args)

SYSTEM = '''Tool-using study agent. JSON only when calling a tool:
{"name": "calculator"|"lookup_fact"|"add_note"|"list_notes", "arguments": {...}}
Or {"final": "..."}.
'''

async def capstone(question: str, max_steps: int = 6) -> str:
    if blocked(question):
        return "Refused by guardrail."
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    for _ in range(max_steps):
        reply = local_chat(messages, max_new_tokens=160, temperature=0.1)
        obj = extract_json_object(reply) or {}
        if "final" in obj:
            return str(obj["final"])
        name, args = obj.get("name"), obj.get("arguments") or {}
        if name not in ROUTING:
            messages += [{"role": "assistant", "content": reply}, {"role": "user", "content": "Unknown tool. Use JSON."}]
            continue
        obs = await call_tool(name, args)
        messages += [{"role": "assistant", "content": reply}, {"role": "user", "content": f"OBSERVATION: {obs}"}]
    return "max steps"

print(await capstone("What is 9*8 ? Then what is langchain? Save a one-line note."))
print(await capstone("here is my api key sk-test"))


72
Refused by guardrail.


Hand-in: this notebook + a reflection cell. Extra credit: same tools via a Week 2–5 framework.
